In [0]:
import os
import sys
import json
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()
vault.get_storage_client("datacopsadls")

spark = SparkSession.getActiveSession()

key = "fs.azure.account.auth.type.datacopsadls.dfs.core.windows.net"
print(f"[INFO] ADLS 인증 방식: {spark.conf.get(key, 'NOT SET')}")

In [0]:
ACCOUNT      = "datacopsadls"
BASE_SILVER  = f"abfss://silver@{ACCOUNT}.dfs.core.windows.net"
BASE_MASTER  = f"abfss://master@{ACCOUNT}.dfs.core.windows.net"

def list_silver_domains() -> list:
    """silver 컨테이너에서 도메인 목록 반환"""
    try:
        return [
            f.path.rstrip("/").split("/")[-1]
            for f in dbutils.fs.ls(BASE_SILVER)
            if f.isDir() and not f.name.startswith("_")
        ]
    except Exception as e:
        print(f"[WARN] 도메인 목록 조회 실패: {e}")
        return []

domains = list_silver_domains()
print(f"[INFO] Silver 도메인 목록: {domains}")

In [0]:
import threading
import time

# 현재 실행 중인 스트리밍 쿼리 추적
active_domains = set()
queries = []
queries_lock = threading.Lock()

def start_domain_stream(domain: str):
    """도메인 스트리밍 시작"""
    silver_path = f"{BASE_SILVER}/{domain}"
    master_path = f"{BASE_MASTER}/{domain}"
    checkpoint  = f"{BASE_MASTER}/_checkpoints/{domain}"

    query = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", f"{BASE_MASTER}/_schema/{domain}")
        .load(silver_path)
        .drop("_row_hash", "_merged_at")
        .withColumn("_merged_at", F.current_timestamp())
        .writeStream
        .format("parquet")
        .option("checkpointLocation", checkpoint)
        .outputMode("append")
        .start(master_path)
    )

    with queries_lock:
        queries.append((domain, query))
        active_domains.add(domain)

    print(f"[OK] {domain} 스트리밍 시작")


def watch_new_domains(interval_sec=60):
    """주기적으로 Silver에 새 도메인 생겼는지 확인"""
    while True:
        try:
            current = set(list_silver_domains())
            new_domains = current - active_domains
            for domain in new_domains:
                print(f"[NEW] 새 도메인 감지: {domain} → 스트리밍 추가")
                start_domain_stream(domain)
        except Exception as e:
            print(f"[WARN] 도메인 감지 오류: {e}")
        time.sleep(interval_sec)


print("[OK] 도메인 감지 스레드 함수 정의 완료")

In [0]:
# 초기 도메인 스트리밍 시작
for domain in list_silver_domains():
    start_domain_stream(domain)

# 새 도메인 감지 스레드 시작 (60초마다 체크)
watcher = threading.Thread(target=watch_new_domains, args=(60,), daemon=True)
watcher.start()

print(f"\n[실행 중] {len(active_domains)}개 도메인 감지 중")
for name in active_domains:
    print(f"  - {name}")

spark.streams.awaitAnyTermination()